# 4-5 人の入れ替わりを地図にする・4-6 クラスター分析で23区をタイプ分けする / Reference notebook
`LANG` を選んで、すべてのセルを実行します。 / Choose `LANG` and run all cells.

In [ ]:
LANG = "ja"   # "ja" / "en"
BASE_URL = "https://raw.githubusercontent.com/rekishi-data/ai-python-data-analysis/main/data/"

In [ ]:
import os, subprocess, urllib.request
subprocess.run("pip install -q geopandas", shell=True)
for f in ['tokyo23_wards.geojson', 'mesh250_mobility_tokyo23.csv', 'ward_indicators_tokyo23.csv']:
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE_URL + f, f)
        except Exception as e: print(f, "をアップロードしてください / please upload", e)
if not os.path.exists("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True)

In [ ]:
import numpy as np
def mesh_sw(code):
    c=str(code); lat=int(c[0:2])/1.5; lon=int(c[2:4])+100
    lat+=int(c[4])*5/60; lon+=int(c[5])*7.5/60
    lat+=int(c[6])*0.5/60; lon+=int(c[7])*0.75/60
    dlat,dlon=0.5/60,0.75/60
    for d in c[8:]:
        dlat/=2; dlon/=2; k=int(d)
        if k in (3,4): lat+=dlat
        if k in (2,4): lon+=dlon
    return lat,lon,dlat,dlon
def mesh_center(code):
    lat,lon,dlat,dlon=mesh_sw(code); return lat+dlat/2, lon+dlon/2
def mesh_polygon(code):
    from shapely.geometry import box
    lat,lon,dlat,dlon=mesh_sw(code); return box(lon,lat,lon+dlon,lat+dlat)


In [ ]:
import sys, os, re, json, glob, numpy as np, pandas as pd, geopandas as gpd, matplotlib
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
JP="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"; fm.fontManager.addfont(JP)
plt.rcParams.update({"font.family":fm.FontProperties(fname=JP).get_name(),"font.size":8,"savefig.dpi":300,"axes.spines.top":False,"axes.spines.right":False})
EN={"千代田区":"Chiyoda","中央区":"Chuo","港区":"Minato","新宿区":"Shinjuku","文京区":"Bunkyo","台東区":"Taito","墨田区":"Sumida","江東区":"Koto","品川区":"Shinagawa","目黒区":"Meguro","大田区":"Ota","世田谷区":"Setagaya","渋谷区":"Shibuya","中野区":"Nakano","杉並区":"Suginami","豊島区":"Toshima","北区":"Kita","荒川区":"Arakawa","板橋区":"Itabashi","練馬区":"Nerima","足立区":"Adachi","葛飾区":"Katsushika","江戸川区":"Edogawa"}
W=gpd.read_file("tokyo23_wards.geojson")
W6=W.to_crs(6677)
C5=["#f0f0f0","#c8c8c8","#969696","#636363","#252525"]
def read_estat(path):
    return pd.read_csv(path,encoding="shift_jis",skiprows=[1],dtype=str,low_memory=False)
def num(s): return pd.to_numeric(s.replace({"*":np.nan,"-":0}),errors="coerce")
def assign_ward(df,col="mesh_code"):
    cen=[mesh_center(str(c)) for c in df[col]]
    g=gpd.GeoDataFrame(df.copy(),geometry=gpd.points_from_xy([b for a,b in cen],[a for a,b in cen]),crs=4326)
    g["lat"]=[a for a,b in cen]; g["lon"]=[b for a,b in cen]
    j=gpd.sjoin(g,W[["ward_code","ward_ja","geometry"]],predicate="within")
    return pd.DataFrame(j.drop(columns=["geometry","index_right"]))
def meshgdf(df,col="mesh_code"):
    return gpd.GeoDataFrame(df.copy(),geometry=[mesh_polygon(str(c)) for c in df[col]],crs=4326).to_crs(6677)
def choro(ax,g,val,bins,labs,title,na_label=None,legend=True):
    g=g.copy(); g["cls"]=pd.cut(g[val],bins,labels=False,right=False)
    W6.plot(ax=ax,facecolor="white",edgecolor="none")
    gg=g[g.cls.notna()]; gg.plot(ax=ax,color=[C5[int(c)] for c in gg.cls],edgecolor="none")
    W6.boundary.plot(ax=ax,color="black",lw=0.35); ax.set_axis_off()
    if legend:
        h=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(C5,labs)]
        if na_label: h.append(Patch(fc="white",ec="black",lw=0.4,label=na_label))
        ax.legend(handles=h,title=title,loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.3,title_fontsize=7,frameon=False)
def gridlines(ax,im):
    nr,nc=im.get_array().shape
    ax.set_xticks(np.arange(-0.5,nc,1),minor=True); ax.set_yticks(np.arange(-0.5,nr,1),minor=True)
    ax.grid(which="minor",color="black",linewidth=0.8); ax.tick_params(which="minor",length=0)
    for s in ax.spines.values(): s.set_visible(False)


## 4-5 人の入れ替わりと通勤・通学 / Residential mobility and commuting

In [ ]:
J=LANG=="ja"; out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
d=pd.read_csv("mesh250_mobility_tokyo23.csv",dtype={"mesh_code":str})
d["known"]=d.same_address+d.moved; d=d[d.known>=100].copy()
d["r_moved"]=d.moved/d.known*100
d["out_commuters"]=d.commuters-d.work_at_home; d["r_bike"]=d.mode_bicycle/d.out_commuters*100
g=meshgdf(d)
fig,ax=plt.subplots(figsize=(4.5,3.4))
choro(ax,g,"r_moved",[0,20,25,30,40,101],["20%未満","20〜25%","25〜30%","30〜40%","40%以上"] if J else ["< 20%","20–25%","25–30%","30–40%","≥ 40%"],"5年以内に移り住んだ人" if J else "Moved in within 5 years","回答100人未満" if J else "< 100 responses")
fig.savefig(f"{out}/fig4-5-1_mesh_moved.png",bbox_inches="tight"); plt.show()
w=d.groupby("ward_ja")[["same_address","moved","pop5plus"]].sum()
w["rm"]=w.moved/(w.same_address+w.moved)*100; w["unk"]=(1-(w.same_address+w.moved)/w.pop5plus)*100
w=w.sort_values("rm")
fig,axs=plt.subplots(1,2,figsize=(4.5,3.4),sharey=True)
yl=[k if J else EN[k] for k in w.index]
axs[0].barh(yl,w.rm,color="#555555",edgecolor="black",lw=0.5); axs[1].barh(yl,w.unk,color="#bbbbbb",edgecolor="black",lw=0.5)
for i,v in enumerate(w.rm): axs[0].text(v+0.5,i,f"{v:.1f}",va="center",fontsize=6)
for i,v in enumerate(w.unk): axs[1].text(v+0.5,i,f"{v:.1f}",va="center",fontsize=6)
axs[0].set_xlabel("移り住んだ人の割合（%）" if J else "Moved in within 5 yrs (%)",fontsize=7); axs[1].set_xlabel("「不詳」の割合（%）" if J else "'Unknown' responses (%)",fontsize=7)
axs[0].set_xlim(0,50); axs[1].set_xlim(0,50); axs[0].tick_params(labelsize=6.5); axs[1].tick_params(labelsize=6.5)
fig.tight_layout(); fig.savefig(f"{out}/fig4-5-2_ward_moved_unknown.png",bbox_inches="tight"); plt.show()
fig,ax=plt.subplots(figsize=(4.5,3.4))
choro(ax,g.dropna(subset=["r_bike"]),"r_bike",[0,10,15,20,30,101],["10%未満","10〜15%","15〜20%","20〜30%","30%以上"] if J else ["< 10%","10–15%","15–20%","20–30%","≥ 30%"],"通勤・通学で自転車を使う人" if J else "Commuting by bicycle","回答100人未満" if J else "< 100 responses")
fig.savefig(f"{out}/fig4-5-3_mesh_bicycle.png",bbox_inches="tight"); plt.show()
print("ok")


## 4-6 クラスター分析 / Cluster analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from matplotlib.patches import Patch
J=LANG=="ja"; out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
F=pd.read_csv("ward_indicators_tokyo23.csv").set_index("ward_ja")
Z=StandardScaler().fit_transform(F)
ks=range(2,8); sil=[]; ine=[]
for k in ks:
    km=KMeans(k,n_init=50,random_state=0).fit(Z); sil.append(silhouette_score(Z,km.labels_)); ine.append(km.inertia_)
fig,axs=plt.subplots(1,2,figsize=(4.5,1.9))
axs[0].plot(list(ks),ine,"k-o",ms=3,mfc="white"); axs[0].set_xlabel("クラスターの数" if J else "Number of clusters"); axs[0].set_ylabel("クラスター内のばらつき" if J else "Within-cluster SS",fontsize=7)
axs[1].plot(list(ks),sil,"k-o",ms=3,mfc="white"); axs[1].set_xlabel("クラスターの数" if J else "Number of clusters"); axs[1].set_ylabel("シルエット係数" if J else "Silhouette score",fontsize=7)
for ax in axs: ax.tick_params(labelsize=6.5)
fig.tight_layout(); fig.savefig(f"{out}/fig4-6-1_choose_k.png",bbox_inches="tight"); plt.show()
km=KMeans(4,n_init=50,random_state=0).fit(Z); F["cl"]=km.labels_
zdf=pd.DataFrame(Z,index=F.index,columns=F.columns[:-1]); prof=zdf.groupby(F.cl).mean()
order=prof["daynight_log"].sort_values(ascending=False).index.tolist()
NJ=["都心業務型","副都心・繁華街型","山の手住宅型","下町・郊外住宅型"]; NE=["Central business","Sub-center/nightlife","Inner residential","Outer residential"]
# name by order of day-night ratio then single_hh: assign manually by membership
def name_of(c):
    mem=set(F.index[F.cl==c])
    if "千代田区" in mem: return 0
    if "新宿区" in mem: return 1
    if "渋谷区" in mem: return 2
    return 3
F["type"]=[name_of(c) for c in F.cl]
prof=zdf.groupby(F.type).mean().sort_index()
IJ={"density":"人口密度","aging":"高齢化率","foreign":"外国人の割合","daynight_log":"昼夜間人口比","single_hh":"単身世帯の割合","child_hh":"6歳未満の子がいる世帯","moved":"移り住んだ人の割合","bicycle":"自転車通勤・通学"}
IE={"density":"Density","aging":"Aged 65+","foreign":"Foreign residents","daynight_log":"Day-night ratio","single_hh":"One-person hh","child_hh":"Hh with child <6","moved":"Moved in 5 yrs","bicycle":"Bicycle commuting"}
fig,ax=plt.subplots(figsize=(4.5,2.2))
im=ax.imshow(prof.values,cmap="Greys",vmin=-2,vmax=2.5,aspect="auto")
for i in range(prof.shape[0]):
    for j in range(prof.shape[1]):
        v=prof.values[i,j]; ax.text(j,i,f"{v:+.1f}",ha="center",va="center",fontsize=6.5,color="white" if v>1.2 else "black")
ax.set_xticks(range(prof.shape[1]),[(IJ if J else IE)[c] for c in prof.columns],rotation=40,ha="right",rotation_mode="anchor",fontsize=6.5)
ax.set_yticks(range(4),[(NJ if J else NE)[i] for i in prof.index],fontsize=6.8); gridlines(ax,im)
fig.colorbar(im,ax=ax,shrink=0.8).ax.tick_params(labelsize=6)
fig.tight_layout(); fig.savefig(f"{out}/fig4-6-3_profile.png",bbox_inches="tight"); plt.show()
Wm=W6.set_index("ward_ja").join(F[["type"]])
STY=[("#252525",""),("#969696",""),("white","////"),("white","")]
fig,ax=plt.subplots(figsize=(4.5,3.4))
for t,(fc,hat) in enumerate(STY):
    Wm[Wm["type"]==t].plot(ax=ax,facecolor=fc,edgecolor="black",lw=0.6,hatch=hat)
halo=[pe.withStroke(linewidth=1.8,foreground="white")]
for k,r in Wm.iterrows():
    pt=r.geometry.representative_point(); ax.text(pt.x,pt.y,k if J else EN[k],ha="center",va="center",fontsize=5.5,path_effects=halo)
ax.set_axis_off()
ax.legend(handles=[Patch(fc=fc,ec="black",hatch=hat,lw=0.5,label=(NJ if J else NE)[t]) for t,(fc,hat) in enumerate(STY)],loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.5,frameon=False,title="区のタイプ" if J else "Ward type",title_fontsize=7)
fig.savefig(f"{out}/fig4-6-2_cluster_map.png",bbox_inches="tight"); plt.show()
if J:
    print([round(s,3) for s in sil]); print(F.groupby("type").apply(lambda x:"・".join(x.index)).to_string()); print(prof.round(2).to_string())
